# Robust Model Evaluation

Proper baselines, time-based validation, lag feature engineering, confidence intervals, and statistical significance tests.

Addresses the limitations of random train/test splits on time-series data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier
from scipy import stats

plt.style.use('seaborn-v0_8-darkgrid')

df = pd.read_csv('datasets/Preprocessed_csv.csv', parse_dates=['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# Aggregate to daily (one row per day)
daily = df.groupby('Date').agg({
    'compound': 'mean',
    'positive_sentiment_score': 'mean',
    'negative_sentiment_score': 'mean',
    'neutral_sentiment_score': 'mean',
    'importance_coefficient': 'mean',
    'importance_coefficient_normalized': 'mean',
    'favorite_count': 'sum',
    'reply_count': 'sum',
    'retweet_count': 'sum',
    'sentiment_type': 'mean',
    'Open': 'first',
    'Close': 'first',
    'Volume': 'first',
    'daily_return': 'first',
    'historical_volatility': 'first',
    'target': 'first'
}).reset_index()

print(f"Daily samples: {len(daily)}")
print(f"Date range: {daily['Date'].min().date()} → {daily['Date'].max().date()}")

## 1. Baselines

Any model must beat these to be useful.

In [ ]:
majority_class = daily['target'].mode()[0]
majority_acc = (daily['target'] == majority_class).mean()

# Momentum: predict yesterday's direction
momentum_preds = daily['target'].shift(1)
momentum_acc = (momentum_preds == daily['target']).dropna().mean()

# Anti-momentum (mean reversion): predict opposite of yesterday
anti_momentum_preds = 1 - daily['target'].shift(1)
anti_momentum_acc = (anti_momentum_preds == daily['target']).dropna().mean()

# Random baseline (expected value)
random_acc = 0.50

baselines = pd.DataFrame({
    'Baseline': ['Random (coin flip)', 'Majority class', 'Momentum (repeat yesterday)', 'Mean reversion (flip yesterday)'],
    'Accuracy': [random_acc, majority_acc, momentum_acc, anti_momentum_acc]
}).sort_values('Accuracy', ascending=False)

baselines['Accuracy'] = baselines['Accuracy'].map('{:.2%}'.format)
baselines

## 2. Lag Feature Engineering

Create features from *previous days* — the only information available at prediction time.

In [ ]:
# Lag features: sentiment and engagement from previous days
lag_features = ['compound', 'positive_sentiment_score', 'negative_sentiment_score',
                'importance_coefficient', 'favorite_count', 'retweet_count']

for feat in lag_features:
    for lag in [1, 2, 3]:
        daily[f'{feat}_lag{lag}'] = daily[feat].shift(lag)

# Rolling sentiment features
daily['compound_ma3'] = daily['compound'].rolling(3).mean()
daily['compound_ma7'] = daily['compound'].rolling(7).mean()
daily['compound_std3'] = daily['compound'].rolling(3).std()
daily['sentiment_momentum'] = daily['compound'] - daily['compound'].shift(1)

# Market features from previous days (no leakage)
daily['return_lag1'] = daily['daily_return'].shift(1)
daily['return_lag2'] = daily['daily_return'].shift(2)
daily['volatility_lag1'] = daily['historical_volatility'].shift(1)
daily['volume_change'] = daily['Volume'].pct_change()

# Drop rows with NaN from lagging
daily_clean = daily.dropna().reset_index(drop=True)
print(f"Samples after lag engineering: {len(daily_clean)}")
print(f"Features created: {len([c for c in daily_clean.columns if 'lag' in c or 'ma' in c or 'std' in c or 'momentum' in c])}")

## 3. Time-Based Train/Test Split

**Critical**: random splits on time-series cause data leakage — future information leaks into training.
We use chronological splits: train on the past, test on the future.

In [ ]:
# Define feature sets
base_sentiment = ['compound', 'positive_sentiment_score', 'negative_sentiment_score',
                   'importance_coefficient', 'sentiment_type']
base_engagement = ['favorite_count', 'retweet_count', 'reply_count']
base_market = ['Open', 'Volume', 'historical_volatility']

lag_cols = [c for c in daily_clean.columns if 'lag' in c]
rolling_cols = ['compound_ma3', 'compound_ma7', 'compound_std3', 'sentiment_momentum',
                'return_lag1', 'return_lag2', 'volatility_lag1', 'volume_change']

# Feature set with lag features (no leakage)
all_features = lag_cols + rolling_cols + base_market

# Time-based split: 80% train, 20% test (chronological)
split_idx = int(len(daily_clean) * 0.8)
split_date = daily_clean.iloc[split_idx]['Date']

X = daily_clean[all_features]
y = daily_clean['target']

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train: {len(X_train)} samples ({daily_clean['Date'].iloc[0].date()} → {daily_clean['Date'].iloc[split_idx-1].date()})")
print(f"Test:  {len(X_test)} samples ({daily_clean['Date'].iloc[split_idx].date()} → {daily_clean['Date'].iloc[-1].date()})")
print(f"Features: {len(all_features)}")
print(f"\nTrain target dist: {y_train.value_counts(normalize=True).to_dict()}")
print(f"Test target dist:  {y_test.value_counts(normalize=True).to_dict()}")

## 4. Model Training with Time-Based Validation

In [ ]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                              subsample=0.8, colsample_bytree=0.8, random_state=42,
                              eval_metric='logloss'),
    'AdaBoost': AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=3),
        n_estimators=200, learning_rate=0.1, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)

    results[name] = {'Accuracy': acc, 'F1': f1, 'AUC-ROC': auc, 'model': model, 'preds': y_pred, 'proba': y_proba}
    print(f"\n{'='*50}")
    print(f"{name} (time-based split)")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, digits=3))

In [ ]:
# Summary comparison
summary = pd.DataFrame({name: {k: v for k, v in vals.items() if k in ['Accuracy', 'F1', 'AUC-ROC']}
                         for name, vals in results.items()}).T
summary = summary.sort_values('Accuracy', ascending=False)
summary = summary.round(4)

# Add baselines for context
summary.loc['Baseline: Majority'] = {'Accuracy': majority_acc, 'F1': 0.0, 'AUC-ROC': 0.5}
summary.loc['Baseline: Momentum'] = {'Accuracy': momentum_acc, 'F1': np.nan, 'AUC-ROC': 0.5}
summary = summary.sort_values('Accuracy', ascending=False)

print("Model Comparison (time-based split, lag features)")
summary

## 5. Walk-Forward Validation

More robust than a single split — simulates real-world deployment where the model is periodically retrained on expanding history.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

walk_forward_results = {name: [] for name in models}

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

    for name, model_template in models.items():
        # Clone model for fresh training
        from sklearn.base import clone
        model = clone(model_template)
        model.fit(X_tr, y_tr)
        acc = accuracy_score(y_te, model.predict(X_te))
        walk_forward_results[name].append(acc)

print("Walk-Forward Validation (5-fold TimeSeriesSplit)")
print("=" * 55)
for name, accs in walk_forward_results.items():
    mean_acc = np.mean(accs)
    std_acc = np.std(accs)
    print(f"{name:20s}  mean={mean_acc:.2%} ± {std_acc:.2%}  folds={[f'{a:.2%}' for a in accs]}")

print(f"\n{'Majority baseline':20s}  {majority_acc:.2%}")
print(f"{'Momentum baseline':20s}  {momentum_acc:.2%}")

## 6. Confidence Intervals (Bootstrap)

In [ ]:
def bootstrap_ci(y_true, y_pred, metric_fn=accuracy_score, n_boot=2000, ci=0.95):
    """Bootstrap confidence interval for a classification metric."""
    scores = []
    rng = np.random.RandomState(42)
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        scores.append(metric_fn(np.array(y_true)[idx], np.array(y_pred)[idx]))
    alpha = (1 - ci) / 2
    return np.percentile(scores, [alpha * 100, (1 - alpha) * 100])

print("95% Confidence Intervals (2000 bootstrap iterations)")
print("=" * 60)

for name, vals in results.items():
    acc = vals['Accuracy']
    ci_low, ci_high = bootstrap_ci(y_test, vals['preds'])
    f1_ci_low, f1_ci_high = bootstrap_ci(y_test, vals['preds'], metric_fn=f1_score)
    print(f"\n{name}:")
    print(f"  Accuracy: {acc:.2%}  95% CI: [{ci_low:.2%}, {ci_high:.2%}]")
    print(f"  F1:       {vals['F1']:.2%}  95% CI: [{f1_ci_low:.2%}, {f1_ci_high:.2%}]")

print(f"\nBaseline accuracy: {majority_acc:.2%}")
print("Models whose CI lower bound exceeds the baseline have statistically significant improvement.")

## 7. Statistical Significance (McNemar's Test)

Tests whether two models make significantly different errors — not just different accuracy numbers.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

def mcnemar_test(y_true, preds_a, preds_b, name_a, name_b):
    """McNemar's test comparing two classifiers."""
    correct_a = (preds_a == y_true)
    correct_b = (preds_b == y_true)

    # Contingency table: [both_right, a_right_b_wrong], [a_wrong_b_right, both_wrong]
    n00 = ((~correct_a) & (~correct_b)).sum()
    n01 = ((~correct_a) & (correct_b)).sum()
    n10 = ((correct_a) & (~correct_b)).sum()
    n11 = ((correct_a) & (correct_b)).sum()

    table = np.array([[n11, n10], [n01, n00]])
    result = mcnemar(table, exact=True)
    sig = '***' if result.pvalue < 0.001 else '**' if result.pvalue < 0.01 else '*' if result.pvalue < 0.05 else ''
    print(f"{name_a} vs {name_b}: p={result.pvalue:.4f} {sig}")
    return result

print("McNemar's Test (pairwise model comparison)")
print("=" * 50)
print("H0: Both models make the same errors")
print("* p<0.05  ** p<0.01  *** p<0.001\n")

model_names = list(results.keys())
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        mcnemar_test(np.array(y_test),
                     results[model_names[i]]['preds'],
                     results[model_names[j]]['preds'],
                     model_names[i], model_names[j])

## 8. Feature Importance Analysis

In [ ]:
# Use the best model's feature importances
best_name = max(results, key=lambda k: results[k]['Accuracy'])
best_model = results[best_name]['model']

importances = pd.Series(best_model.feature_importances_, index=all_features)
top_20 = importances.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 8))
top_20.sort_values().plot(kind='barh', ax=ax, color='#3498db', edgecolor='black', linewidth=0.5)
ax.set_title(f'Top 20 Feature Importances ({best_name})', fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

# Categorize features
sentiment_imp = importances[[c for c in all_features if any(s in c for s in ['compound', 'sentiment', 'importance'])]].sum()
engagement_imp = importances[[c for c in all_features if any(s in c for s in ['favorite', 'retweet', 'reply'])]].sum()
market_imp = importances[[c for c in all_features if any(s in c for s in ['Open', 'Volume', 'return', 'volatility', 'volume_change'])]].sum()

print(f"\nImportance by category:")
print(f"  Sentiment features: {sentiment_imp:.2%}")
print(f"  Engagement features: {engagement_imp:.2%}")
print(f"  Market features: {market_imp:.2%}")

## 9. Impact of Random vs Time-Based Split

Demonstrating why the validation method matters.

In [ ]:
from sklearn.model_selection import train_test_split

# Random split (flawed for time-series)
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.2, random_state=42)

comparison = []
for name, model_template in models.items():
    from sklearn.base import clone

    # Random split
    m_rand = clone(model_template)
    m_rand.fit(X_train_rand, y_train_rand)
    acc_rand = accuracy_score(y_test_rand, m_rand.predict(X_test_rand))

    # Time-based split
    acc_time = results[name]['Accuracy']

    comparison.append({
        'Model': name,
        'Random Split': f'{acc_rand:.2%}',
        'Time-Based Split': f'{acc_time:.2%}',
        'Inflation': f'{(acc_rand - acc_time):.2%}'
    })

comp_df = pd.DataFrame(comparison)
print("Accuracy: Random Split vs Time-Based Split")
print("(Inflation = how much data leakage inflates results)\n")
comp_df

## Summary

| Aspect | Method |
|--------|--------|
| **Baselines** | Majority class, momentum, mean reversion, random |
| **Validation** | Chronological 80/20 split + 5-fold walk-forward |
| **Features** | Lag-1/2/3 sentiment, rolling averages, lagged returns (no leakage) |
| **Confidence** | 2000-iteration bootstrap 95% CIs |
| **Significance** | McNemar's test for pairwise model comparison |
| **Leakage audit** | Random vs time-based split comparison quantifies inflation |